# Ekspor Core ML pipeline MuQ saja

Buka dari root **SongMap**; memakai Python `.venv-songformer311` di sibling Section-Annotations (perlu `coremltools`, versi yang dulu dipakai mengekspor head palette6).

Pipeline MuQ + konteks lolos validation dan test (test: 0,8135 / 0,7182 vs produksi 0,8000 / 0,6997). Notebook ini membuat tiga paket Core ML **tanpa encoder**:

| Paket | Input | Presisi |
|---|---|---|
| `MuQFunctionHeadFP32.mlpackage` | `[1,T,2048]` MuQ30+MuQ120 | FP32 (referensi) |
| `MuQFunctionHeadPalette6.mlpackage` | sama | palette6, konfigurasi sama dengan head app sekarang |
| `MuQSectionContextFP32.mlpackage` | `[N,2048]` + `[N,6]` | FP32 |

Lalu verifikasi pada 32 lagu validation dari cache fitur asli. Gerbang (tetap sebelum hasil):
- FP32 head + konteks Core ML: label identik dengan PyTorch, selisih probabilitas akhir maks. 1e-4.
- Palette6 head + konteks Core ML: akurasi dan macro F1 masing-masing turun maks. 0,01 dari PyTorch.

Encoder MuQ Core ML (bank canonical) belum dijalankan di sini; itu tahap berikutnya.

In [ ]:
from pathlib import Path
from datetime import datetime
import subprocess, json
assert Path('scripts/export_muq_pipeline.py').is_file(), 'Buka notebook dari root SongMap'
SONGSTRUCT_ROOT=Path('../Section-Annotations').resolve()
RUN_PYTHON=SONGSTRUCT_ROOT/'.venv-songformer311/bin/python'
assert RUN_PYTHON.is_file(), f'Python tidak ditemukan: {RUN_PYTHON}'
subprocess.run([str(RUN_PYTHON),'-c',"import sys,torch,coremltools,x_transformers; print(sys.executable); print('torch',torch.__version__,'coremltools',coremltools.__version__)"],check=True)
CONTEXT=sorted(Path('outputs.local').glob('stream-ablation-w120-*'))[-1].resolve()/'context'/'muq'
assert (CONTEXT/'best.pt').is_file() and (CONTEXT/'summary.json').is_file(), f'Konteks MuQ tidak ditemukan: {CONTEXT}'
stamp=datetime.now().strftime('%Y%m%d-%H%M%S')
EXPORT=Path('outputs.local')/('muq-coreml-export-'+stamp)
VERIFY=Path('outputs.local')/('muq-coreml-verify-'+stamp)
CACHE=Path('outputs.local')/'variant-context-cache'
print('Konteks MuQ:',CONTEXT); print('Ekspor:',EXPORT.resolve()); print('Verifikasi:',VERIFY.resolve())

In [ ]:
def run(*arguments,name,folder):
    folder.parent.mkdir(parents=True,exist_ok=True)
    log=folder.parent/(folder.name+'-'+name+'.log')
    command=[str(RUN_PYTHON),'-u','scripts/export_muq_pipeline.py',*arguments]
    with log.open('w',encoding='utf-8') as handle:
        process=subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        try:
            for line in process.stdout:
                if 'it/s]' in line or 'passes/s]' in line or 'ops/s]' in line: continue  # converter progress bars
                print(line,end='');handle.write(line);handle.flush()
            status=process.wait()
        except KeyboardInterrupt:
            process.terminate();process.wait();raise
    if status:
        raise RuntimeError(f'Gagal ({status}); lihat {log.resolve()}')

run('export','--songstruct-root',str(SONGSTRUCT_ROOT),'--context-dir',str(CONTEXT),'--output-dir',str(EXPORT),name='export',folder=EXPORT)

In [ ]:
run('verify','--songstruct-root',str(SONGSTRUCT_ROOT),'--context-dir',str(CONTEXT),'--export-dir',str(EXPORT),
    '--cache-dir',str(CACHE),'--output-dir',str(VERIFY),name='verify',folder=VERIFY)
print('Bagikan:',(EXPORT/'export_report.json').resolve(),'dan',(VERIFY/'verification.json').resolve())

Bila verifikasi lolos, tahap berikutnya: validasi penuh dengan encoder MuQ Core ML (bank canonical palette6 yang sudah ada), lalu probe iPhone tanpa MusicFM untuk mengukur ukuran terpasang, waktu dan memori.